# notch Filtering analysis - JYXFE
etiqueta negra - Patricio rey y sus redonditos de ricota

In [5]:
from pathlib import Path
import sys

# In scripts, use __file__; in notebooks, use current working directory
try:
    current_path = Path(__file__).resolve().parent
except NameError:
    current_path = Path.cwd().resolve()

# Go up until you find the project root where "src" exists
for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        project_root = parent
        break
else:
    raise FileNotFoundError("Project root not found. No parent folder contains 'src'.")

# Add project root to PYTHONPATH if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

# Now import works
from src.modules import tools_EEG_Preprocess as TEEG_PR

Project root: /home/tperezsanchez/Tomas_PS_DissertationKCL2026/Main_project


## Manual inspection one file only

In [ ]:
import numpy as np
from scipy.signal import welch
import matplotlib.pyplot as plt
#1. LOAD NPZ
#
npz_path= "/home/tperezsanchez/Tomas_PS_DissertationKCL2026/Main_project/results/JYXFE/Pre_processing/JYXFE_IN-JYXFE_AMP200_BP0p5-48Hz_NOTCH34p5Hz_NOZSCORE_20260511/npz/JYXFE_100_preproc_full.npz"

data = np.load(npz_path, allow_pickle=True)
#print(data.files) # for checking the keys 
#2. EXTRACT VARIABLES NEEDED
X = data["X"]
fs = float(data["fs"])
channel_names = data["channel_names"]
#print(X.shape)
#print(fs)
#print(channel_names)
#3.EXTRACT A CHANNEL, each 
canal_0 = X[0, :]
canal_1 = X[1, :]
#0 = primera fila = primer canal

#: = todas las columnas = todas las muestras en el tiempo
print(canal_0.shape)
#
#f, Pxx = welch(canal_0, fs=fs, nperseg=int(fs * 4))
#f: vector de frecuencias
#Pxx: potencia estimada en cada frecuencia
#


plt.figure(figsize=(9, 5))

for i in range(X.shape[0]):
    sig = X[i, :]
    f, Pxx = welch(sig, fs=fs, nperseg=int(fs * 4))
    plt.semilogy(f, Pxx, label=str(channel_names[i]))

plt.axvline(0.5, linestyle="--", label="0.5 Hz cutoff")
plt.axvline(48, linestyle="--", label="48 Hz cutoff")
plt.xlim(0, min(120, fs / 2))
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD")
plt.title("Welch PSD of preprocessed EEG")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
## all files

In [ ]:
npz_path_process = "/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_testALL_25032026/XB47Y_54_preproc_full.npz"

TEEG.plot_psd_from_npz_1_12(
    npz_path_1= npz_path_process,
    channel=1
)

In [ ]:
df_ok = df_peaks[df_peaks["status"] == "ok"].copy()

file_summary = (
    df_ok.groupby("file")
    .agg(
        n_channels=("channel_idx", "count"),
        mean_peak_freq_hz=("peak_freq_hz", "mean"),
        std_peak_freq_hz=("peak_freq_hz", "std"),
        mean_peak_ratio=("peak_to_baseline_ratio", "mean"),
        pct_near_target=("near_target", "mean"),
        pct_high_peak=("high_peak", "mean"),
    )
    .reset_index()
)

file_summary["pct_near_target"] *= 100
file_summary["pct_high_peak"] *= 100

print(file_summary.head())